<a href="https://colab.research.google.com/github/niharnandala/ntsb-safety-review-agent/blob/main/notebooks/00_setup_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi

Fri Nov 13 06:52:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q git+https://github.com/Lexsi-Labs/AgentTune.git
!pip install -q auditkit

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Requested pytablewriter from https://files.pythonhosted.org/packages/c6/79/62ff8772fa8b30e992053f7fd06cd5c9a484088732e11e70db4f421de2b0/pytablewriter-0.47.0-py2.py3-none-any.whl (from lm-eva

In [3]:
import agenttune
import auditkit
print("both loaded ✅")

from agenttune import agentic, decide, backends
print("all 3 parts found ✅")

both loaded ✅
all 3 parts found ✅


In [7]:
from google.colab import userdata
#we are logging into our hugging face account directly using our private hugging face key so that we can import llm models from hugging face
from huggingface_hub import login

login(userdata.get("HF_TOKEN"))

In [ ]:
prompt_text = tok.apply_chat_template(
    messages, tools=[get_recommendation], add_generation_prompt=True, tokenize=False
)
print(prompt_text)
print("\nTool in prompt?", "get_recommendation" in prompt_text)

<BOS_TOKEN><|START_OF_TURN_TOKEN|><|SYSTEM_TOKEN|># System Preamble
You are in contextual safety mode. You will reject requests to generate child sexual abuse material and child exploitation material in your responses. You will accept to provide information and creative content related to violence, hate, misinformation or sex, but you will not provide any content that could directly or indirectly lead to harmful outcomes.

Your information cutoff date is June 2024.

You have been trained on data in English, Dutch, French, Italian, Portuguese, Romanian, Spanish, Czech, Polish, Ukrainian, Russian, Greek, German, Danish, Swedish, Norwegian, Catalan, Galician, Welsh, Irish, Basque, Croatian, Latvian, Lithuanian, Slovak, Slovenian, Estonian, Finnish, Hungarian, Serbian, Bulgarian, Arabic, Persian, Urdu, Turkish, Maltese, Hebrew, Hindi, Marathi, Bengali, Gujarati, Punjabi, Tamil, Telugu, Nepali, Tagalog, Malay, Indonesian, Vietnamese, Javanese, Khmer, Thai, Lao, Chinese, Burmese, Japanese, K

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
#tokenizer: this will turn our text into numbers for model to understand
#Auto-model-For-Casual-lm:Its main job is to load an LLM so it can predict and generate the next words for your prompt.

model_id = "CohereLabs/tiny-aya-global"
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map="auto")

print("loaded ✅  GPU memory used:", round(torch.cuda.memory_allocated() / 1e9, 1), "GB")

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

loaded ✅  GPU memory used: 6.7 GB


In [11]:
def get_recommendation(case_id: str) -> str:
  # this is a tool named get_recommendation..we are testing the ai using this tool wther it can call or not
  #Below comments in the doc string is read by LLm .it gives LLm info about what this tool is about!!
    """Get what NTSB asked for in a case, its priority, and how many letters it has.

    Args:
        case_id: The case ID, for example CASE-0001
    """
    return "not connected yet"



In [12]:
import json
from transformers.utils import get_json_schema
from agenttune.agentic.rollout_engines.rollout_factory import DEFAULT_TOOLS_FALLBACK_PROMPT
from agenttune.agentic.rollout_engines.tool_call_parse import extract_tool_calls_from_text

# 1. Turn our Python function into a tool description (JSON)
tool_schema = get_json_schema(get_recommendation)

print("Tool Schema : ",tool_schema)

# 2. Put the tool menu into the instructions, in AgentTune's format
#this is the instructions that will be given to the LLM as Prompt
#In the prompt we also added what tools it can call
system_text = (
    "You assist an NTSB safety analyst. Use tools to look things up before answering.\n\n"
    + DEFAULT_TOOLS_FALLBACK_PROMPT.format(tools=json.dumps([tool_schema], indent=2))
)

print("DEFAULT_TOOLS_FALLBACK_PROMPT ",DEFAULT_TOOLS_FALLBACK_PROMPT)
print("System text : ", system_text)
#this is the message we give to LLM
#it has Promt + Our question
messages = [
    {"role": "system", "content": system_text},
    {"role": "user", "content": "Review case CASE-0001."},
]

# 3. Ask the model (no tools= this time, because the menu is already in the text)
inputs = tok.apply_chat_template(
    messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
).to(model.device)
out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
reply = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print("MODEL REPLY:\n", reply)

# 4. Can AgentTune's reader find a tool call in it?
print("\nAGENTTUNE SEES:", extract_tool_calls_from_text(reply))

Tool Schema :  {'type': 'function', 'function': {'name': 'get_recommendation', 'description': 'Get what NTSB asked for in a case, its priority, and how many letters it has.', 'parameters': {'type': 'object', 'properties': {'case_id': {'type': 'string', 'description': 'The case ID, for example CASE-0001'}}, 'required': ['case_id']}, 'return': {'type': 'string'}}}
DEFAULT_TOOLS_FALLBACK_PROMPT  You can call these tools:
{tools}

To call tools, reply with only a JSON list, one object per call, and no other text: [{{"tool_name": "<name>", "parameters": {{"<argument>": <value>}}}}]. Use a list even for one call. A user turn that starts with "Tool result" is the output of your call, not a new message from the user: use it to answer the user's original question as plain text.
System text :  You assist an NTSB safety analyst. Use tools to look things up before answering.

You can call these tools:
[
  {
    "type": "function",
    "function": {
      "name": "get_recommendation",
      "descri